# 03 — Modelling
**QM640 Data Analytics Capstone** | Nandini Nagaraj | Walsh College | Spring 2026

---

## Purpose
This notebook trains and evaluates all classification models for the Interim and Final Reports.
Run  first.

## Contents
| Section | Content |
|---|---|
| 1 | Install & load |
| 2 | Logistic Regression — Set A (individual indicators) |
| 3 | Logistic Regression — Set B (composite index) |
| 4 | Confusion matrices |
| 5 | Random Forest — placeholder |
| 6 | XGBoost — placeholder |
| 7 | Model comparison (McNemar's test) — placeholder |
| 8 | SHAP feature importance — placeholder |

> Sections marked **placeholder** are ready for you to complete for the Final Report.
> The logistic regression baseline (Sections 2–4) is fully implemented.

---
## Section 1: Install Libraries & Load Data

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost shap matplotlib seaborn
print("✅ Libraries installed.")

In [ ]:
import pandas as pd
import numpy as np
import warnings, os
warnings.filterwarnings("ignore")

if os.path.exists("dpi_state_clean.csv"):
    df = pd.read_csv("dpi_state_clean.csv")
    print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} cols")
else:
    # ── PLACEHOLDER: Update URL ──
    GITHUB_RAW_URL = "https://raw.githubusercontent.com/NandiniJu/dpi-capstone-qm640/main/dpi_state_data.csv"
    df = pd.read_csv(GITHUB_RAW_URL)
    if "per_capita_income_inr" in df.columns:
        df.rename(columns={"per_capita_income_inr": "per_capita_income"}, inplace=True)
    from sklearn.preprocessing import MinMaxScaler
    df["log_per_capita_income"] = np.log(df["per_capita_income"])
    scaler = MinMaxScaler()
    df["digital_adoption_index"] = scaler.fit_transform(
        df[["upi_txn_per_capita", "internet_penetration", "aadhaar_coverage"]]
    ).mean(axis=1)
    print(f"✅ Loaded from GitHub: {df.shape[0]} rows × {df.shape[1]} cols")

df.head(3)

---
## Section 2 & 3: Logistic Regression — Set A and Set B

Trains logistic regression on both feature sets and prints **Table 8** values.

> **PLACEHOLDER:** After running, copy the printed metrics into Table 8 in your Word document.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

FEATURES_A = [
    'upi_txn_per_capita', 'internet_penetration', 'aadhaar_coverage',
    'literacy_rate', 'log_per_capita_income', 'urbanization_rate'
]
FEATURES_B = [
    'digital_adoption_index',
    'literacy_rate', 'log_per_capita_income', 'urbanization_rate'
]
TARGET = 'performance_category'

X_A = df[FEATURES_A].values
X_B = df[FEATURES_B].values
y   = df[TARGET].values

# Stratified 80/20 split (same random_state for both sets — fair comparison)
X_trA, X_teA, y_tr, y_te = train_test_split(X_A, y, test_size=0.2, random_state=42, stratify=y)
X_trB, X_teB, _,   _    = train_test_split(X_B, y, test_size=0.2, random_state=42, stratify=y)

sc_A, sc_B = StandardScaler(), StandardScaler()
X_trA, X_teA = sc_A.fit_transform(X_trA), sc_A.transform(X_teA)
X_trB, X_teB = sc_B.fit_transform(X_trB), sc_B.transform(X_teB)

lr_A = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=500, random_state=42)
lr_B = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=500, random_state=42)
lr_A.fit(X_trA, y_tr)
lr_B.fit(X_trB, y_tr)

def metrics(y_true, y_pred, y_proba, label):
    return {
        'Model / Feature Set': label,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 3),
        'ROC-AUC':   round(roc_auc_score(y_true, y_proba), 3),
    }

results = pd.DataFrame([
    metrics(y_te, lr_A.predict(X_teA), lr_A.predict_proba(X_teA)[:,1], 'LR — Set A (individual indicators)'),
    metrics(y_te, lr_B.predict(X_teB), lr_B.predict_proba(X_teB)[:,1], 'LR — Set B (composite index)'),
])

print("\n" + "="*70)
print("TABLE 8 VALUES — copy these into your Word document")
print("="*70)
print(results.to_string(index=False))
print("="*70)
print("\n⬆️  Replace the placeholder numbers in Table 8 with the values above.")

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (X_te, label) in zip(
    axes,
    [(X_teA, 'Set A: Individual Indicators'), (X_teB, 'Set B: Composite Index')]
):
    model = lr_A if 'A' in label else lr_B
    cm = confusion_matrix(y_te, model.predict(X_te))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Low (0)', 'High (1)'],
        yticklabels=['Low (0)', 'High (1)'],
        linewidths=1, ax=ax, cbar=False
    )
    ax.set_title(f'LR — {label}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted', labelpad=8)
    ax.set_ylabel('Actual', labelpad=8)

plt.suptitle(
    'Figure 5. Confusion Matrices — Logistic Regression Baseline (Test Set)',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('figures/figure5_confusion_matrix.png')
plt.show()
print("✅ Saved: figures/figure5_confusion_matrix.png")

---
## Section 5: Random Forest

> **PLACEHOLDER — complete for Final Report.**
> Add your random forest training code below using the same FEATURES_A / FEATURES_B split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

FEATURES_A = [
    "upi_txn_per_capita", "internet_penetration", "aadhaar_coverage",
    "literacy_rate", "log_per_capita_income", "urbanization_rate"
]
TARGET = "performance_category"

# ── PLACEHOLDER: Add your grid search and training code here ──
# Example structure:
# param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, None], "min_samples_split": [2, 5]}
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=cv, scoring="roc_auc")
# rf.fit(X_trainA, y_tr)
# print("Best params:", rf.best_params_)
# print("CV AUC:", round(rf.best_score_, 3))
print("⬆️  Implement Random Forest training above for the Final Report.")

---
## Section 6: XGBoost

> **PLACEHOLDER — complete for Final Report.**

In [ ]:
import xgboost as xgb

# ── PLACEHOLDER: Add your XGBoost training code here ──
# Example structure:
# param_grid = {"learning_rate": [0.05, 0.1], "n_estimators": [100, 200],
#               "max_depth": [3, 5], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]}
# xgb_model = GridSearchCV(xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
#                          param_grid, cv=cv, scoring="roc_auc")
# xgb_model.fit(X_trainA, y_tr)
# print("Best params:", xgb_model.best_params_)
# print("CV AUC:", round(xgb_model.best_score_, 3))
print("⬆️  Implement XGBoost training above for the Final Report.")

---
## Section 7: Model Comparison — McNemar's Test (RQ3)

> **PLACEHOLDER — complete after all three models are trained.**
> McNemar's test compares whether two classifiers make significantly different errors on the same test set.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# ── PLACEHOLDER: Run after all three models produce test-set predictions ──
# Example (LR vs RF):
# from sklearn.metrics import confusion_matrix
# import numpy as np
#
# pred_lr = lr_A.predict(X_teA)
# pred_rf = rf.predict(X_teA)        # from Section 5
#
# # Build contingency table
# b = np.sum((pred_lr == y_te) & (pred_rf != y_te))  # LR correct, RF wrong
# c = np.sum((pred_lr != y_te) & (pred_rf == y_te))  # LR wrong, RF correct
# contingency = np.array([[0, b], [c, 0]])            # off-diagonal only
#
# result = mcnemar(contingency, exact=False, correction=True)
# print(f"McNemar chi2 = {result.statistic:.3f}, p = {result.pvalue:.4f}")
# print("Significant difference" if result.pvalue < 0.05 else "No significant difference")
print("⬆️  Implement McNemar test above after all models are trained.")

---
## Section 8: SHAP Feature Importance (RQ2)

> **PLACEHOLDER — complete after Random Forest and XGBoost are trained.**
> SHAP values provide global and local feature attribution for RQ2.

In [ ]:
import shap

# ── PLACEHOLDER: Run after XGBoost model is trained ──
# Global SHAP summary plot (XGBoost):
# explainer = shap.TreeExplainer(xgb_model.best_estimator_)
# shap_values = explainer.shap_values(X_teA)
# shap.summary_plot(shap_values, X_teA, feature_names=FEATURES_A, show=False)
# import matplotlib.pyplot as plt
# plt.title("Figure X. SHAP Summary Plot — XGBoost (Set A)")
# plt.tight_layout()
# plt.savefig("figures/shap_summary_xgb.png", dpi=200, bbox_inches="tight")
# plt.show()
#
# Local waterfall for a single prediction:
# shap.waterfall_plot(shap.Explanation(values=shap_values[0],
#                     base_values=explainer.expected_value,
#                     data=X_teA[0], feature_names=FEATURES_A))
print("⬆️  Implement SHAP analysis above for the Final Report.")

---
## ✅ Notebook Complete

### Outputs
| Output | Status |
|---|---|
| Table 8 metrics (LR Set A & B) | ✅ Complete — copy into Word doc |
| Confusion matrices | ✅ Saved as  |
| Random Forest results | ⬜ Placeholder — complete for Final Report |
| XGBoost results | ⬜ Placeholder — complete for Final Report |
| McNemar's test (RQ3) | ⬜ Placeholder — complete after all models trained |
| SHAP analysis (RQ2) | ⬜ Placeholder — complete after XGBoost trained |

> Upload all three notebooks to GitHub under  once complete.